# Benford-priority Rich Feature SVM Full-Data (CASIA + Columbia)

This notebook trains three **CNN-free** SVM variants on full CASIA + Columbia:

1. `Benford13-only`
2. `Benford++`
3. `BenfordRich`

Artifacts are written to a versioned run folder for later benchmark and integration.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

REQUIRED_PACKAGES = [
    ('cv2', 'opencv-python-headless==4.10.0.84'),
    ('joblib', 'joblib==1.5.3'),
    ('numpy', 'numpy==2.0.2'),
    ('pandas', 'pandas==2.2.3'),
    ('sklearn', 'scikit-learn==1.6.1'),
    ('matplotlib', 'matplotlib==3.10.0'),
    ('seaborn', 'seaborn==0.13.2'),
    ('PIL', 'Pillow==11.1.0'),
    ('tqdm', 'tqdm==4.67.1'),
    ('kaggle', 'kaggle==1.6.17'),
]


def ensure_packages(packages):
    for import_name, pip_spec in packages:
        if importlib.util.find_spec(import_name) is None:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_spec])


ensure_packages(REQUIRED_PACKAGES)

import cv2
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
print('Python      :', platform.python_version())
print('NumPy       :', np.__version__)
print('scikit-learn:', __import__('sklearn').__version__)
print('Joblib      :', joblib.__version__)


In [ ]:
# ===== PUBLIC CONFIG =====
DATA_ROOT = Path('/content/datasets')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/T07_colab/artifacts/benford_rich_artifacts')
SEED = 42

CASIA_KAGGLE_SLUG = 'divg07/casia-20-image-tampering-detection-dataset'
COLUMBIA_KAGGLE_SLUG = 'shriya0/columbia'

AUTO_MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = '/content/drive'
AUTO_UPLOAD_KAGGLE_JSON = True
AUTO_DOWNLOAD_DATASETS = True
AUTO_REUSE_SPLIT = True

DRIVE_KAGGLE_JSON_CANDIDATES = [
    '/content/drive/MyDrive/T07_colab/kaggle.json',
    '/content/drive/MyDrive/kaggle.json',
    '/content/drive/MyDrive/.kaggle/kaggle.json',
    '/content/drive/MyDrive/T07_colab/keys/kaggle.json',
]
REFERENCE_ARTIFACT_ROOT_CANDIDATES = [
    '/content/drive/MyDrive/T07_colab/artifacts/auto_ifake_casia_columbia',
    '/content/auto_ifake_casia_columbia_artifacts',
]
LOCAL_REPO_ROOT_CANDIDATES = [
    '/content/T07FakeMediaDetect',
    '/content/drive/MyDrive/T07FakeMediaDetect',
    '/content/drive/MyDrive/T07_colab/T07FakeMediaDetect',
]

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp'}
LABEL_MAPPING = {'authentic': 0, 'forged': 1}
FEATURE_SCHEMA_VERSION = 'benford_rich_v1'
MIN_IMAGES_TOTAL = 200
HOLDOUT_RATIO = 0.2
VAL_RATIO_FROM_TRAINVAL = 0.15
RUN_TS = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
RUN_DIR = None

print('DATA_ROOT     =', DATA_ROOT)
print('ARTIFACT_ROOT =', ARTIFACT_ROOT)


In [ ]:
# Self-contained helper module for Colab execution
# This cell mirrors benford_rich_feature_utils.py so the notebook can run as a single uploaded file.

from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np

BENFORD_CHI_SCALE = 1000.0
PATCH_SIZE = 128
BENFORD_METRIC_NAMES = [
    "digit_1",
    "digit_2",
    "digit_3",
    "digit_4",
    "digit_5",
    "digit_6",
    "digit_7",
    "digit_8",
    "digit_9",
    "chi_square_scaled",
    "ks",
    "mad",
    "mse",
]
FEATURE_GROUPS = {
    "benford13_only": [f"gray_global_{name}" for name in BENFORD_METRIC_NAMES],
    "benford_plus_extra": [f"y_global_{name}" for name in BENFORD_METRIC_NAMES]
    + [f"cb_global_{name}" for name in BENFORD_METRIC_NAMES]
    + [f"cr_global_{name}" for name in BENFORD_METRIC_NAMES]
    + [f"gray_laplacian_{name}" for name in BENFORD_METRIC_NAMES]
    + [
        "patch_chi_mean",
        "patch_chi_std",
        "patch_chi_max",
        "patch_chi_p90",
        "patch_ks_mean",
        "patch_ks_std",
        "patch_ks_max",
        "patch_ks_p90",
        "patch_mad_mean",
        "patch_mad_std",
        "patch_mad_max",
        "patch_mad_p90",
        "patch_mse_mean",
        "patch_mse_std",
        "patch_mse_max",
        "patch_mse_p90",
    ],
    "jpeg_blockiness_6": [
        "jpeg_row_boundary_mean",
        "jpeg_row_boundary_std",
        "jpeg_row_boundary_max",
        "jpeg_col_boundary_mean",
        "jpeg_col_boundary_std",
        "jpeg_boundary_ratio",
    ],
    "noise_stats_8": [
        "noise_gauss_mean",
        "noise_gauss_std",
        "noise_gauss_max",
        "noise_gauss_p90",
        "noise_median_mean",
        "noise_median_std",
        "noise_median_max",
        "noise_median_p90",
    ],
    "edge_gradient_8": [
        "edge_sobel_mean",
        "edge_sobel_std",
        "edge_sobel_max",
        "edge_sobel_p90",
        "edge_canny_density",
        "edge_laplacian_variance",
        "edge_boundary_grad_ratio",
        "edge_canny_boundary_ratio",
    ],
    "color_inconsistency_8": [
        "color_cb_std",
        "color_cr_std",
        "color_cb_var",
        "color_cr_var",
        "color_abs_corr_y_cb",
        "color_abs_corr_y_cr",
        "color_abs_corr_cb_cr",
        "color_chroma_to_luma_var_ratio",
    ],
}
FEATURE_GROUPS["benford_plus"] = (
    FEATURE_GROUPS["benford13_only"] + FEATURE_GROUPS["benford_plus_extra"]
)
FEATURE_GROUPS["benford_rich"] = (
    FEATURE_GROUPS["benford_plus"]
    + FEATURE_GROUPS["jpeg_blockiness_6"]
    + FEATURE_GROUPS["noise_stats_8"]
    + FEATURE_GROUPS["edge_gradient_8"]
    + FEATURE_GROUPS["color_inconsistency_8"]
)
FEATURE_WIDTHS = {
    "benford13_only": len(FEATURE_GROUPS["benford13_only"]),
    "benford_plus": len(FEATURE_GROUPS["benford_plus"]),
    "benford_rich": len(FEATURE_GROUPS["benford_rich"]),
}

BENFORD_EXPECTED = np.array(
    [np.log10(1 + 1.0 / d) for d in range(1, 10)], dtype=np.float32
)


def ensure_uint8_gray(channel: np.ndarray) -> np.ndarray:
    if channel.dtype == np.uint8:
        return channel
    arr = np.asarray(channel, dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if arr.size == 0:
        return np.zeros((1, 1), dtype=np.uint8)
    min_val = float(arr.min())
    max_val = float(arr.max())
    if max_val <= min_val:
        return np.zeros(arr.shape, dtype=np.uint8)
    scaled = (arr - min_val) / (max_val - min_val)
    return np.clip(np.round(scaled * 255.0), 0, 255).astype(np.uint8)


def resize_if_needed(image_bgr: np.ndarray) -> np.ndarray:
    if image_bgr.shape[0] > 1000:
        return cv2.resize(
            image_bgr, (0, 0), fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA
        )
    return image_bgr


def load_feature_views(image_path: str):
    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        return None
    image_bgr = resize_if_needed(image_bgr)
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    ycrcb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
    y = ycrcb[:, :, 0]
    cr = ycrcb[:, :, 1]
    cb = ycrcb[:, :, 2]
    lap = cv2.Laplacian(gray, cv2.CV_32F)
    return {
        "bgr": image_bgr,
        "gray": gray,
        "y": y,
        "cb": cb,
        "cr": cr,
        "gray_laplacian": ensure_uint8_gray(np.abs(lap)),
    }


def extract_first_digits_from_dct_array(channel: np.ndarray):
    arr = np.asarray(channel, dtype=np.float32)
    if arr.ndim != 2:
        return []
    h, w = arr.shape
    first_digits = []
    for i in range(0, h - 8, 8):
        for j in range(0, w - 8, 8):
            coeffs = cv2.dct(arr[i : i + 8, j : j + 8]).flatten()[1:]
            for value in coeffs:
                abs_val = abs(float(value))
                if abs_val >= 1:
                    digit = int(str(int(abs_val))[0])
                    if 1 <= digit <= 9:
                        first_digits.append(digit)
    return first_digits


def benford13_from_channel(channel: np.ndarray) -> np.ndarray:
    digits = extract_first_digits_from_dct_array(channel)
    if not digits:
        return np.zeros(13, dtype=np.float32)
    counts = np.array([digits.count(d) for d in range(1, 10)], dtype=np.float32)
    total = float(counts.sum())
    observed = counts / total
    chi_sq = float(np.sum((observed - BENFORD_EXPECTED) ** 2 / BENFORD_EXPECTED) * total)
    ks = float(np.max(np.abs(np.cumsum(observed) - np.cumsum(BENFORD_EXPECTED))))
    mad = float(np.mean(np.abs(observed - BENFORD_EXPECTED)))
    mse = float(np.mean((observed - BENFORD_EXPECTED) ** 2))
    return np.concatenate(
        [observed, np.array([chi_sq / BENFORD_CHI_SCALE, ks, mad, mse], dtype=np.float32)]
    ).astype(np.float32)


def safe_summary(values) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float32)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.zeros(4, dtype=np.float32)
    return np.array(
        [
            float(arr.mean()),
            float(arr.std()),
            float(arr.max()),
            float(np.percentile(arr, 90)),
        ],
        dtype=np.float32,
    )


def patch_benford_stats(gray: np.ndarray, patch_size: int = PATCH_SIZE) -> np.ndarray:
    gray = np.asarray(gray, dtype=np.uint8)
    h, w = gray.shape
    if h < patch_size or w < patch_size:
        patches = [gray]
    else:
        patches = [
            gray[y : y + patch_size, x : x + patch_size]
            for y in range(0, h - patch_size + 1, patch_size)
            for x in range(0, w - patch_size + 1, patch_size)
        ] or [gray]
    patch_stats = [benford13_from_channel(patch)[-4:] for patch in patches]
    patch_arr = np.asarray(patch_stats, dtype=np.float32)
    if patch_arr.size == 0:
        return np.zeros(16, dtype=np.float32)
    features = []
    for col_idx in range(4):
        features.extend(safe_summary(patch_arr[:, col_idx]).tolist())
    return np.asarray(features, dtype=np.float32)


def boundary_ratio_from_map(value_map: np.ndarray, step: int = 8) -> float:
    arr = np.asarray(value_map, dtype=np.float32)
    if arr.ndim != 2 or arr.size == 0:
        return 0.0
    row_idx = np.arange(arr.shape[0])
    col_idx = np.arange(arr.shape[1])
    boundary_mask = ((row_idx % step) == (step - 1))[:, None] | (
        (col_idx % step) == (step - 1)
    )[None, :]
    interior_mask = ~boundary_mask
    boundary_vals = arr[boundary_mask]
    interior_vals = arr[interior_mask]
    boundary_mean = float(boundary_vals.mean()) if boundary_vals.size else 0.0
    interior_mean = float(interior_vals.mean()) if interior_vals.size else 0.0
    return float(boundary_mean / (interior_mean + 1e-6))


def jpeg_blockiness_features(gray: np.ndarray) -> np.ndarray:
    gray_f = np.asarray(gray, dtype=np.float32)
    row_diffs = np.abs(np.diff(gray_f, axis=0))
    col_diffs = np.abs(np.diff(gray_f, axis=1))
    row_boundary_idx = np.arange(7, row_diffs.shape[0], 8)
    col_boundary_idx = np.arange(7, col_diffs.shape[1], 8)
    row_boundary_vals = (
        row_diffs[row_boundary_idx, :].mean(axis=1)
        if row_boundary_idx.size
        else np.array([], dtype=np.float32)
    )
    col_boundary_vals = (
        col_diffs[:, col_boundary_idx].mean(axis=0)
        if col_boundary_idx.size
        else np.array([], dtype=np.float32)
    )
    row_non_idx = np.setdiff1d(np.arange(row_diffs.shape[0]), row_boundary_idx)
    col_non_idx = np.setdiff1d(np.arange(col_diffs.shape[1]), col_boundary_idx)
    row_non_vals = (
        row_diffs[row_non_idx, :].mean(axis=1)
        if row_non_idx.size
        else np.array([], dtype=np.float32)
    )
    col_non_vals = (
        col_diffs[:, col_non_idx].mean(axis=0)
        if col_non_idx.size
        else np.array([], dtype=np.float32)
    )
    boundary_sum = (
        float(row_boundary_vals.mean()) if row_boundary_vals.size else 0.0
    ) + (float(col_boundary_vals.mean()) if col_boundary_vals.size else 0.0)
    nonboundary_sum = (
        float(row_non_vals.mean()) if row_non_vals.size else 0.0
    ) + (float(col_non_vals.mean()) if col_non_vals.size else 0.0)
    return np.asarray(
        [
            float(row_boundary_vals.mean()) if row_boundary_vals.size else 0.0,
            float(row_boundary_vals.std()) if row_boundary_vals.size else 0.0,
            float(row_boundary_vals.max()) if row_boundary_vals.size else 0.0,
            float(col_boundary_vals.mean()) if col_boundary_vals.size else 0.0,
            float(col_boundary_vals.std()) if col_boundary_vals.size else 0.0,
            float(boundary_sum / (nonboundary_sum + 1e-6)),
        ],
        dtype=np.float32,
    )


def noise_stats_features(gray: np.ndarray) -> np.ndarray:
    gray_f = np.asarray(gray, dtype=np.float32)
    gauss = cv2.GaussianBlur(gray_f, (5, 5), 0)
    median = cv2.medianBlur(np.asarray(gray, dtype=np.uint8), 5).astype(np.float32)
    return np.concatenate(
        [
            safe_summary(np.abs(gray_f - gauss).reshape(-1)),
            safe_summary(np.abs(gray_f - median).reshape(-1)),
        ]
    ).astype(np.float32)


def edge_gradient_features(gray: np.ndarray) -> np.ndarray:
    gray_f = np.asarray(gray, dtype=np.float32)
    sobel_x = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)
    sobel_mag = np.sqrt((sobel_x ** 2) + (sobel_y ** 2))
    canny = cv2.Canny(np.asarray(gray, dtype=np.uint8), 100, 200)
    lap_var = float(cv2.Laplacian(np.asarray(gray, dtype=np.uint8), cv2.CV_64F).var())
    return np.asarray(
        [
            float(sobel_mag.mean()),
            float(sobel_mag.std()),
            float(sobel_mag.max()) if sobel_mag.size else 0.0,
            float(np.percentile(sobel_mag, 90)) if sobel_mag.size else 0.0,
            float((canny > 0).mean()),
            lap_var,
            boundary_ratio_from_map(sobel_mag, step=8),
            boundary_ratio_from_map(canny.astype(np.float32), step=8),
        ],
        dtype=np.float32,
    )


def safe_abs_corr(a: np.ndarray, b: np.ndarray) -> float:
    a_flat = np.asarray(a, dtype=np.float32).reshape(-1)
    b_flat = np.asarray(b, dtype=np.float32).reshape(-1)
    if a_flat.size == 0 or b_flat.size == 0:
        return 0.0
    if float(a_flat.std()) < 1e-6 or float(b_flat.std()) < 1e-6:
        return 0.0
    corr = np.corrcoef(a_flat, b_flat)[0, 1]
    if not np.isfinite(corr):
        return 0.0
    return float(abs(corr))


def color_inconsistency_features(y: np.ndarray, cb: np.ndarray, cr: np.ndarray) -> np.ndarray:
    y_f = np.asarray(y, dtype=np.float32)
    cb_f = np.asarray(cb, dtype=np.float32)
    cr_f = np.asarray(cr, dtype=np.float32)
    return np.asarray(
        [
            float(cb_f.std()),
            float(cr_f.std()),
            float(cb_f.var()),
            float(cr_f.var()),
            safe_abs_corr(y_f, cb_f),
            safe_abs_corr(y_f, cr_f),
            safe_abs_corr(cb_f, cr_f),
            float((cb_f.var() + cr_f.var()) / (y_f.var() + 1e-6)),
        ],
        dtype=np.float32,
    )


def extract_feature_sets(image_path: str):
    views = load_feature_views(image_path)
    if views is None:
        return {
            "benford13_only": np.zeros(FEATURE_WIDTHS["benford13_only"], dtype=np.float32),
            "benford_plus": np.zeros(FEATURE_WIDTHS["benford_plus"], dtype=np.float32),
            "benford_rich": np.zeros(FEATURE_WIDTHS["benford_rich"], dtype=np.float32),
        }
    gray13 = benford13_from_channel(views["gray"])
    y13 = benford13_from_channel(views["y"])
    cb13 = benford13_from_channel(views["cb"])
    cr13 = benford13_from_channel(views["cr"])
    lap13 = benford13_from_channel(views["gray_laplacian"])
    patch16 = patch_benford_stats(views["gray"], patch_size=PATCH_SIZE)
    benford13 = gray13.astype(np.float32)
    benford_plus = np.concatenate([gray13, y13, cb13, cr13, lap13, patch16]).astype(
        np.float32
    )
    benford_rich = np.concatenate(
        [
            benford_plus,
            jpeg_blockiness_features(views["gray"]),
            noise_stats_features(views["gray"]),
            edge_gradient_features(views["gray"]),
            color_inconsistency_features(views["y"], views["cb"], views["cr"]),
        ]
    ).astype(np.float32)
    return {
        "benford13_only": np.nan_to_num(benford13, nan=0.0, posinf=0.0, neginf=0.0),
        "benford_plus": np.nan_to_num(benford_plus, nan=0.0, posinf=0.0, neginf=0.0),
        "benford_rich": np.nan_to_num(benford_rich, nan=0.0, posinf=0.0, neginf=0.0),
    }


def feature_summary_rows(feature_map: dict[str, np.ndarray]):
    rows = []
    for name, matrix in feature_map.items():
        rows.append(
            {
                "feature_set": name,
                "width": int(matrix.shape[1]),
                "min": float(matrix.min()),
                "max": float(matrix.max()),
                "mean": float(matrix.mean()),
                "std": float(matrix.std()),
            }
        )
    return rows


In [ ]:
import hashlib


def count_images(root: Path) -> int:
    if not root.exists():
        return 0
    return sum(1 for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def ensure_drive_mounted() -> bool:
    if not AUTO_MOUNT_DRIVE:
        return False
    if Path('/content/drive/MyDrive').exists():
        return True
    try:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT)
        return Path('/content/drive/MyDrive').exists()
    except Exception as exc:
        print('Drive mount failed:', exc)
        return False


def setup_artifact_root() -> Path:
    global RUN_DIR
    active_root = ARTIFACT_ROOT if ensure_drive_mounted() else Path('/content/benford_rich_artifacts')
    active_root.mkdir(parents=True, exist_ok=True)
    RUN_DIR = active_root / f'run_{RUN_TS}'
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    print('RUN_DIR =', RUN_DIR)
    return RUN_DIR


def setup_kaggle_auth_if_needed() -> bool:
    kaggle_dir = Path('/root/.kaggle')
    kaggle_json = kaggle_dir / 'kaggle.json'
    if kaggle_json.exists():
        return True
    if ensure_drive_mounted():
        for candidate in DRIVE_KAGGLE_JSON_CANDIDATES:
            path = Path(candidate)
            if path.exists() and path.is_file():
                kaggle_dir.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, kaggle_json)
                os.chmod(kaggle_json, 0o600)
                print('Kaggle auth copied from Drive:', path)
                return True
    if not AUTO_UPLOAD_KAGGLE_JSON:
        return False
    try:
        from google.colab import files
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            kaggle_dir.mkdir(parents=True, exist_ok=True)
            kaggle_json.write_bytes(uploaded['kaggle.json'])
            os.chmod(kaggle_json, 0o600)
            print('Kaggle auth configured at', kaggle_json)
            return True
    except Exception as exc:
        print('Kaggle auth upload failed:', exc)
    return kaggle_json.exists()


def kaggle_download_dataset_if_needed(slug: str, target_dir: Path, min_existing_images: int = 50) -> None:
    target_dir.mkdir(parents=True, exist_ok=True)
    if count_images(target_dir) >= min_existing_images:
        print(f'Skip download {slug}: enough images already in {target_dir}')
        return
    if not AUTO_DOWNLOAD_DATASETS:
        raise RuntimeError(f'Dataset missing and AUTO_DOWNLOAD_DATASETS=False: {slug}')
    if not setup_kaggle_auth_if_needed():
        raise RuntimeError('Kaggle auth unavailable. Cannot download datasets.')
    cmd = [sys.executable, '-m', 'kaggle', 'datasets', 'download', '-d', slug, '-p', str(target_dir), '--unzip', '--force']
    subprocess.check_call(cmd)
    print(f'Download and extract done: {target_dir}')


def infer_dataset_source(path: Path) -> str:
    p = str(path).lower().replace('\\', '/')
    if '/casia' in p:
        return 'CASIA'
    if '/columbia' in p or '/4cam_auth/' in p or '/4cam_splc/' in p:
        return 'COLUMBIA'
    return 'UNKNOWN'


def infer_label_from_path(path: Path):
    p = str(path).lower().replace('\\', '/')
    if '/au/' in p:
        return LABEL_MAPPING['authentic']
    if '/tp/' in p:
        return LABEL_MAPPING['forged']
    if '/4cam_auth/4cam_auth/' in p:
        return LABEL_MAPPING['authentic']
    if '/4cam_splc/4cam_splc/' in p:
        return LABEL_MAPPING['forged']
    return None


def is_auxiliary_mask_file(path: Path) -> bool:
    parts = {part.lower() for part in path.parts}
    name = path.name.lower()
    return bool(parts & {'mask', 'masks', 'edgemask', 'probe_mask', 'donor_mask'}) or any(k in name for k in ('_mask', '_edgemask', 'mask_'))


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    hasher = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()


def quick_image_valid(path: Path):
    try:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            return False, 'cv2_imread_none'
        return True, ''
    except Exception as exc:
        return False, str(exc)


def build_manifest_with_audit(root: Path):
    rows = []
    bad_rows = []
    unlabeled_rows = []
    skipped_auxiliary = 0
    for file_path in root.rglob('*'):
        if not file_path.is_file() or file_path.suffix.lower() not in IMAGE_EXTS:
            continue
        if is_auxiliary_mask_file(file_path):
            skipped_auxiliary += 1
            continue
        label = infer_label_from_path(file_path)
        source = infer_dataset_source(file_path)
        if label is None:
            unlabeled_rows.append({'path': str(file_path), 'dataset_source': source})
            continue
        if file_path.stat().st_size <= 0:
            bad_rows.append({'path': str(file_path), 'reason': 'zero_byte', 'dataset_source': source, 'label': int(label)})
            continue
        ok, reason = quick_image_valid(file_path)
        if not ok:
            bad_rows.append({'path': str(file_path), 'reason': f'corrupted:{reason}', 'dataset_source': source, 'label': int(label)})
            continue
        rows.append({'path': str(file_path), 'label': int(label), 'dataset_source': source, 'sha256': sha256_file(file_path)})
    if not rows:
        raise RuntimeError(f'No labeled valid images found under {root}')
    manifest_raw = pd.DataFrame(rows).drop_duplicates(subset=['path']).reset_index(drop=True)
    bad_df = pd.DataFrame(bad_rows)
    unlabeled_df = pd.DataFrame(unlabeled_rows)
    dup_hash_df = manifest_raw[manifest_raw.duplicated(subset=['sha256'], keep=False)].sort_values('sha256')
    hash_label_nunique = manifest_raw.groupby('sha256')['label'].nunique().reset_index(name='label_nunique')
    conflict_hashes = set(hash_label_nunique[hash_label_nunique['label_nunique'] > 1]['sha256'].tolist())
    dup_conflict_df = dup_hash_df[dup_hash_df['sha256'].isin(conflict_hashes)].copy()
    manifest_df = manifest_raw.sort_values(['dataset_source', 'path']).drop_duplicates(subset=['sha256'], keep='first').sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return manifest_df, manifest_raw, bad_df, dup_hash_df, dup_conflict_df, unlabeled_df, skipped_auxiliary


def safe_stratify_key(df: pd.DataFrame):
    source_label_key = df['dataset_source'].astype(str) + '_' + df['label'].astype(str)
    vc = source_label_key.value_counts()
    if (vc < 2).any():
        label_vc = df['label'].value_counts()
        return None if (label_vc < 2).any() else df['label']
    return source_label_key


def find_latest_reference_run_dir():
    for root_str in REFERENCE_ARTIFACT_ROOT_CANDIDATES:
        root = Path(root_str)
        if not root.exists():
            continue
        run_dirs = [p for p in root.glob('run_*') if p.is_dir()]
        if run_dirs:
            return max(run_dirs, key=lambda p: p.stat().st_mtime)
    return None


def try_apply_reuse_split(manifest_df: pd.DataFrame):
    if not AUTO_REUSE_SPLIT:
        return manifest_df, False, None
    ref_run = find_latest_reference_run_dir()
    if ref_run is None:
        return manifest_df, False, None
    reuse_path = ref_run / 'manifest_split.csv'
    if not reuse_path.exists():
        return manifest_df, False, None
    reuse_df = pd.read_csv(reuse_path)
    if not {'sha256', 'split'}.issubset(set(reuse_df.columns)):
        return manifest_df, False, reuse_path
    mapping_df = reuse_df[['sha256', 'split']].drop_duplicates(subset=['sha256'])
    merged = manifest_df.merge(mapping_df, on='sha256', how='left')
    coverage = float(merged['split'].notna().mean())
    if coverage < 0.99 or not set(merged['split'].dropna().unique()).issubset({'train', 'val', 'holdout'}):
        return manifest_df, False, reuse_path
    manifest_df = manifest_df.copy()
    manifest_df['split'] = merged['split'].values
    return manifest_df, True, reuse_path


def find_local_repo_root():
    for candidate in LOCAL_REPO_ROOT_CANDIDATES:
        path = Path(candidate)
        if path.exists() and path.is_dir():
            return path
    return None


In [ ]:
RUN_DIR = setup_artifact_root()
casia_dir = DATA_ROOT / 'CASIA'
columbia_dir = DATA_ROOT / 'COLUMBIA'
casia_dir.mkdir(parents=True, exist_ok=True)
columbia_dir.mkdir(parents=True, exist_ok=True)
kaggle_download_dataset_if_needed(CASIA_KAGGLE_SLUG, casia_dir, min_existing_images=200)
kaggle_download_dataset_if_needed(COLUMBIA_KAGGLE_SLUG, columbia_dir, min_existing_images=30)

manifest_df, manifest_raw_df, bad_df, dup_hash_df, dup_conflict_df, unlabeled_df, skipped_auxiliary = build_manifest_with_audit(DATA_ROOT)
fail_reasons = []
if len(manifest_df) < MIN_IMAGES_TOTAL:
    fail_reasons.append(f'total_images<{MIN_IMAGES_TOTAL}')
if len(bad_df) > 0:
    fail_reasons.append('has_corrupted_or_zero_byte_images')
if len(dup_conflict_df) > 0:
    fail_reasons.append('duplicate_hash_conflicting_labels')
for src in ['CASIA', 'COLUMBIA']:
    src_df = manifest_df[manifest_df['dataset_source'] == src]
    if src_df.empty:
        fail_reasons.append(f'missing_source:{src}')
        continue
    if set(src_df['label'].unique().tolist()) != {0, 1}:
        fail_reasons.append(f'source_missing_class:{src}')

manifest_full_path = RUN_DIR / 'benford_rich_manifest_full.csv'
manifest_raw_path = RUN_DIR / 'benford_rich_manifest_raw_before_hash_dedup.csv'
quality_report_path = RUN_DIR / 'benford_rich_data_quality_report.json'
manifest_df.to_csv(manifest_full_path, index=False)
manifest_raw_df.to_csv(manifest_raw_path, index=False)
audit_report = {
    'seed': SEED,
    'data_root': str(DATA_ROOT),
    'total_valid_images_before_hash_dedup': int(len(manifest_raw_df)),
    'total_valid_images_after_hash_dedup': int(len(manifest_df)),
    'duplicate_hash_rows_before_dedup': int(len(dup_hash_df)),
    'duplicate_hash_conflicting_label_rows': int(len(dup_conflict_df)),
    'total_bad_images': int(len(bad_df)),
    'total_unlabeled_rows': int(len(unlabeled_df)),
    'skipped_auxiliary_mask_files': int(skipped_auxiliary),
    'label_mapping': LABEL_MAPPING,
    'label_counts': {str(k): int(v) for k, v in manifest_df['label'].value_counts().to_dict().items()},
    'source_counts': {str(k): int(v) for k, v in manifest_df['dataset_source'].value_counts().to_dict().items()},
    'audit_pass_pre_split': len(fail_reasons) == 0,
    'audit_fail_reasons_pre_split': fail_reasons,
}
with open(quality_report_path, 'w', encoding='utf-8') as handle:
    json.dump(audit_report, handle, indent=2)
print('Valid images AFTER hash dedup:', len(manifest_df))
print(manifest_df['dataset_source'].value_counts())
print(manifest_df['label'].value_counts())
if not audit_report['audit_pass_pre_split']:
    raise RuntimeError(f"Data quality audit failed (pre-split): {audit_report['audit_fail_reasons_pre_split']}")


In [ ]:
manifest_df, split_reused, split_reuse_path = try_apply_reuse_split(manifest_df)
if not split_reused:
    stratify_key = safe_stratify_key(manifest_df)
    all_idx = np.arange(len(manifest_df))
    trainval_idx, holdout_idx = train_test_split(all_idx, test_size=HOLDOUT_RATIO, random_state=SEED, stratify=stratify_key)
    trainval_df = manifest_df.iloc[trainval_idx].reset_index(drop=False).rename(columns={'index': 'orig_idx'})
    train_idx_local, val_idx_local = train_test_split(np.arange(len(trainval_df)), test_size=VAL_RATIO_FROM_TRAINVAL, random_state=SEED, stratify=safe_stratify_key(trainval_df))
    train_idx = trainval_df.iloc[train_idx_local]['orig_idx'].values
    val_idx = trainval_df.iloc[val_idx_local]['orig_idx'].values
    split_col = np.array(['train'] * len(manifest_df), dtype=object)
    split_col[val_idx] = 'val'
    split_col[holdout_idx] = 'holdout'
    manifest_df = manifest_df.copy()
    manifest_df['split'] = split_col
holdout_idx = np.where(manifest_df['split'].values == 'holdout')[0]
train_idx = np.where(manifest_df['split'].values == 'train')[0]
val_idx = np.where(manifest_df['split'].values == 'val')[0]
leaks = {
    'train_val_overlap': int(len(set(manifest_df.iloc[train_idx]['sha256']) & set(manifest_df.iloc[val_idx]['sha256']))),
    'train_holdout_overlap': int(len(set(manifest_df.iloc[train_idx]['sha256']) & set(manifest_df.iloc[holdout_idx]['sha256']))),
    'val_holdout_overlap': int(len(set(manifest_df.iloc[val_idx]['sha256']) & set(manifest_df.iloc[holdout_idx]['sha256']))),
}
if any(v > 0 for v in leaks.values()):
    raise RuntimeError(f'Cross-split leakage detected: {leaks}')
manifest_split_path = RUN_DIR / 'manifest_split.csv'
holdout_manifest_path = RUN_DIR / 'benford_rich_holdout_manifest.csv'
manifest_df.to_csv(manifest_split_path, index=False)
manifest_df.iloc[holdout_idx].to_csv(holdout_manifest_path, index=False)
print(manifest_df['split'].value_counts())
print('Saved:', manifest_split_path)
print('Saved:', holdout_manifest_path)


## Feature extraction and training

`Benford13-only` uses grayscale global Benford 13.

`Benford++` adds Y/Cb/Cr global Benford, Laplacian Benford, and patch Benford stats.

`BenfordRich` adds JPEG blockiness, noise, edge, and color inconsistency summaries on top of `Benford++`.


In [ ]:
X_b13, X_plus, X_rich, Y = [], [], [], []
for row in tqdm(manifest_df.itertuples(index=False), total=len(manifest_df), desc='Extracting features'):
    feat_sets = extract_feature_sets(row.path)
    X_b13.append(feat_sets['benford13_only'])
    X_plus.append(feat_sets['benford_plus'])
    X_rich.append(feat_sets['benford_rich'])
    Y.append(int(row.label))
X_b13 = np.asarray(X_b13, dtype=np.float32)
X_plus = np.asarray(X_plus, dtype=np.float32)
X_rich = np.asarray(X_rich, dtype=np.float32)
Y = np.asarray(Y, dtype=np.int64)
assert X_b13.shape[1] == FEATURE_WIDTHS['benford13_only']
assert X_plus.shape[1] == FEATURE_WIDTHS['benford_plus']
assert X_rich.shape[1] == FEATURE_WIDTHS['benford_rich']
assert np.isfinite(X_b13).all() and np.isfinite(X_plus).all() and np.isfinite(X_rich).all()
summary_df = pd.DataFrame(feature_summary_rows({'benford13_only': X_b13, 'benford_plus': X_plus, 'benford_rich': X_rich}))
summary_path = RUN_DIR / 'benford_rich_feature_summary.csv'
summary_df.to_csv(summary_path, index=False)
display(summary_df)
print('Saved:', summary_path)


In [ ]:
trainval_mask = manifest_df['split'].isin(['train', 'val']).values
holdout_mask = manifest_df['split'].values == 'holdout'
y_trainval = Y[trainval_mask]
y_holdout = Y[holdout_mask]
X_trainval_map = {'benford13_only': X_b13[trainval_mask], 'benford_plus': X_plus[trainval_mask], 'benford_rich': X_rich[trainval_mask]}
X_holdout_map = {'benford13_only': X_b13[holdout_mask], 'benford_plus': X_plus[holdout_mask], 'benford_rich': X_rich[holdout_mask]}
param_grid = {'C': [1, 5, 10, 20, 50], 'gamma': ['scale', 0.1, 0.01, 0.001], 'class_weight': [None, 'balanced']}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def train_variant(X_trainval, X_holdout):
    scaler = StandardScaler()
    X_trainval_scaled = scaler.fit_transform(X_trainval)
    X_holdout_scaled = scaler.transform(X_holdout)
    grid = GridSearchCV(SVC(kernel='rbf', probability=True, random_state=SEED), param_grid=param_grid, scoring='f1', cv=cv, n_jobs=-1, verbose=1)
    grid.fit(X_trainval_scaled, y_trainval)
    model = grid.best_estimator_
    probs = model.predict_proba(X_holdout_scaled)[:, 1]
    preds = (probs >= 0.5).astype(int)
    metrics = {
        'accuracy': float(accuracy_score(y_holdout, preds)),
        'precision_forged': float(precision_score(y_holdout, preds, pos_label=1, zero_division=0)),
        'recall_forged': float(recall_score(y_holdout, preds, pos_label=1, zero_division=0)),
        'f1_forged': float(f1_score(y_holdout, preds, pos_label=1, zero_division=0)),
        'roc_auc_forged': float(roc_auc_score(y_holdout, probs)),
        'confusion_matrix_labels_0auth_1forged': confusion_matrix(y_holdout, preds, labels=[0, 1]).tolist(),
        'svm_best_params': grid.best_params_,
        'svm_best_cv_score_f1': float(grid.best_score_),
        'prediction_unique_labels': sorted(pd.Series(preds).unique().tolist()),
    }
    return model, scaler, probs, preds, metrics

trained = {}
for variant_name in ['benford13_only', 'benford_plus', 'benford_rich']:
    trained[variant_name] = {}
    model, scaler, probs, preds, metrics = train_variant(X_trainval_map[variant_name], X_holdout_map[variant_name])
    trained[variant_name].update({'model': model, 'scaler': scaler, 'probs_holdout': probs, 'preds_holdout': preds, 'metrics': metrics})
    print(variant_name, json.dumps(metrics, indent=2))


In [ ]:
artifact_name_map = {
    'benford13_only': ('svm_benford13.pkl', 'scaler_benford13.pkl', 'benford13_holdout_metrics.json'),
    'benford_plus': ('svm_benford_plus.pkl', 'scaler_benford_plus.pkl', 'benford_plus_holdout_metrics.json'),
    'benford_rich': ('svm_benford_rich.pkl', 'scaler_benford_rich.pkl', 'benford_rich_holdout_metrics.json'),
}
metrics_per_model = {}
best_params_per_model = {}
for variant_name, payload in trained.items():
    model_name, scaler_name, metrics_name = artifact_name_map[variant_name]
    joblib.dump(payload['model'], RUN_DIR / model_name)
    joblib.dump(payload['scaler'], RUN_DIR / scaler_name)
    (RUN_DIR / metrics_name).write_text(json.dumps(payload['metrics'], indent=2), encoding='utf-8')
    metrics_per_model[variant_name] = payload['metrics']
    best_params_per_model[variant_name] = payload['metrics']['svm_best_params']

holdout_predictions_df = manifest_df.loc[holdout_mask, ['path', 'label', 'dataset_source', 'split', 'sha256']].copy().reset_index(drop=True)
for variant_name, payload in trained.items():
    holdout_predictions_df[f'prob_{variant_name}'] = payload['probs_holdout']
    holdout_predictions_df[f'pred_{variant_name}'] = payload['preds_holdout']
predictions_path = RUN_DIR / 'benford_rich_holdout_predictions.csv'
holdout_predictions_df.to_csv(predictions_path, index=False)

reference_summary = {}
reference_run = find_latest_reference_run_dir()
if reference_run is not None:
    hybrid_metrics_path = reference_run / 'hybrid_holdout_metrics.json'
    cnn_metrics_path = reference_run / 'selected_cnn_metrics.json'
    if hybrid_metrics_path.exists():
        reference_summary['current_hybrid_holdout'] = json.loads(hybrid_metrics_path.read_text(encoding='utf-8'))
    if cnn_metrics_path.exists():
        reference_summary['current_cnn_holdout'] = json.loads(cnn_metrics_path.read_text(encoding='utf-8'))

compare_rows = []
for variant_name in ['benford13_only', 'benford_plus', 'benford_rich']:
    m = metrics_per_model[variant_name]
    compare_rows.append({'model': variant_name, 'accuracy': m['accuracy'], 'precision_forged': m['precision_forged'], 'recall_forged': m['recall_forged'], 'f1_forged': m['f1_forged'], 'roc_auc_forged': m['roc_auc_forged']})
for ref_name in ['current_hybrid_holdout', 'current_cnn_holdout']:
    if ref_name in reference_summary:
        m = reference_summary[ref_name]
        compare_rows.append({'model': ref_name, 'accuracy': m['accuracy'], 'precision_forged': m['precision_forged'], 'recall_forged': m['recall_forged'], 'f1_forged': m['f1_forged'], 'roc_auc_forged': m['roc_auc_forged']})
compare_df = pd.DataFrame(compare_rows).sort_values(['f1_forged', 'accuracy'], ascending=False).reset_index(drop=True)
compare_path = RUN_DIR / 'benford_rich_compare_table.csv'
compare_df.to_csv(compare_path, index=False)
display(compare_df)

benford13_metrics = metrics_per_model['benford13_only']
benford_plus_metrics = metrics_per_model['benford_plus']
benford_rich_metrics = metrics_per_model['benford_rich']
acceptance_gate = {
    'better_than_benford13': benford_rich_metrics['f1_forged'] >= benford13_metrics['f1_forged'] + 0.05 or benford_rich_metrics['accuracy'] >= benford13_metrics['accuracy'] + 0.05,
    'not_worse_than_benford_plus': benford_rich_metrics['f1_forged'] >= benford_plus_metrics['f1_forged'] and benford_rich_metrics['accuracy'] >= benford_plus_metrics['accuracy'],
    'not_collapsed_to_single_class': len(benford_rich_metrics['prediction_unique_labels']) > 1,
}
acceptance_gate['pass'] = all(acceptance_gate.values())
metadata = {
    'feature_schema_version': FEATURE_SCHEMA_VERSION,
    'label_mapping': LABEL_MAPPING,
    'feature_order': FEATURE_GROUPS['benford_rich'],
    'feature_groups': FEATURE_GROUPS,
    'feature_widths': FEATURE_WIDTHS,
    'patch_size': PATCH_SIZE,
    'resize_rule': 'half_if_height_gt_1000',
    'benford_chi_scale': BENFORD_CHI_SCALE,
    'datasets_used': ['CASIA', 'COLUMBIA'],
    'split_seed': SEED,
    'split_manifest_path': str(manifest_split_path),
    'holdout_manifest_path': str(holdout_manifest_path),
    'quality_report_path': str(quality_report_path),
    'best_params_per_model': best_params_per_model,
    'metrics_per_model': metrics_per_model,
    'reference_summary': reference_summary,
    'acceptance_gate': acceptance_gate,
    'train_environment': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'scikit_learn': __import__('sklearn').__version__, 'joblib': joblib.__version__},
}
metadata_path = RUN_DIR / 'benford_rich_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved:', predictions_path)
print('Saved:', metadata_path)
print('Saved:', compare_path)
print('Acceptance gate:', json.dumps(acceptance_gate, indent=2))


In [ ]:
repo_root = find_local_repo_root()
sample_results = []

def evaluate_manifest_with_model(manifest_path: Path, model, scaler, feature_key: str):
    sample_df = pd.read_csv(manifest_path)
    if not {'path', 'label'}.issubset(set(sample_df.columns)):
        raise RuntimeError(f'{manifest_path} missing path/label columns')
    X_eval = []
    y_eval = sample_df['label'].astype(int).values
    for path_value in tqdm(sample_df['path'].astype(str).tolist(), total=len(sample_df), desc=f'Evaluating {manifest_path.name}'):
        feat_sets = extract_feature_sets(path_value)
        X_eval.append(feat_sets[feature_key])
    X_eval = np.asarray(X_eval, dtype=np.float32)
    probs = model.predict_proba(scaler.transform(X_eval))[:, 1]
    preds = (probs >= 0.5).astype(int)
    return {'sample_set': manifest_path.parent.name, 'sample_size': int(len(sample_df)), 'accuracy': float(accuracy_score(y_eval, preds)), 'precision_forged': float(precision_score(y_eval, preds, pos_label=1, zero_division=0)), 'recall_forged': float(recall_score(y_eval, preds, pos_label=1, zero_division=0)), 'f1_forged': float(f1_score(y_eval, preds, pos_label=1, zero_division=0)), 'roc_auc_forged': float(roc_auc_score(y_eval, probs)) if len(np.unique(y_eval)) > 1 else None}

if repo_root is None:
    print('Skip sample-set evaluation: local repo root not found in Colab.')
else:
    for manifest_candidate in [repo_root / 'sample_sets' / 'casia_same_domain_100' / 'manifest.csv', repo_root / 'sample_sets' / 'external_hf_splicing_mix_100' / 'manifest.csv']:
        if not manifest_candidate.exists():
            print('Skip missing sample manifest:', manifest_candidate)
            continue
        try:
            result = evaluate_manifest_with_model(manifest_candidate, trained['benford_rich']['model'], trained['benford_rich']['scaler'], 'benford_rich')
            sample_results.append(result)
            print(json.dumps(result, indent=2))
        except Exception as exc:
            print(f'Sample-set evaluation failed for {manifest_candidate}: {exc}')
if sample_results:
    sample_results_df = pd.DataFrame(sample_results)
    sample_results_path = RUN_DIR / 'benford_rich_sample_set_eval.json'
    sample_results_df.to_json(sample_results_path, orient='records', indent=2)
    display(sample_results_df)
    print('Saved:', sample_results_path)


## Next step after training

- if `BenfordRich` passes the acceptance gate, it becomes the handcrafted candidate for the next integration phase
- if `BenfordRich` fails but `Benford++` is clearly stronger than `Benford13-only`, keep `Benford++`
- if all three remain weak on the external sample set, stop SVM expansion and prioritize `MUN-first`
